In [2]:
import pandas as pd
import numpy as np
import re # Import thư viện regex để xử lý chuỗi phức tạp
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
# --- 1. Hàm xử lý dữ liệu đặc biệt ---

def convert_price_v2(price_str):
    """
    Chuyển đổi chuỗi giá (ví dụ: '5 tỷ 500 triệu', '5 tỷ', '500 triệu') thành số (VNĐ).
    Trả về NaN nếu không thể chuyển đổi.
    """
    if isinstance(price_str, (int, float)):
        return float(price_str) # Nếu đã là số thì trả về
    if not isinstance(price_str, str):
        return np.nan

    price_str = price_str.lower().strip().replace(',', '') # Chuẩn hóa chuỗi
    total_price = 0
    ty_match = re.search(r'(\d+\.?\d*)\s*tỷ', price_str)
    trieu_match = re.search(r'(\d+\.?\d*)\s*triệu', price_str)

    if ty_match:
        total_price += float(ty_match.group(1)) * 1e9
    if trieu_match:
        total_price += float(trieu_match.group(1)) * 1e6

    # Nếu không tìm thấy 'tỷ' hoặc 'triệu', thử chuyển đổi trực tiếp thành số
    if total_price == 0:
        try:
            # Loại bỏ các ký tự không phải số (trừ dấu chấm thập phân nếu có)
            cleaned_str = re.sub(r'[^\d\.]', '', price_str)
            if cleaned_str: # Kiểm tra nếu chuỗi không rỗng sau khi làm sạch
                 # Giả sử nếu không có tỷ/triệu thì đơn vị là triệu VNĐ (cần xác nhận lại)
                 # Hoặc nếu chắc chắn là VNĐ thì dùng: float(cleaned_str)
                 num_val = float(cleaned_str)
                 # Phỏng đoán: nếu số quá nhỏ, có thể là đơn vị triệu
                 if num_val < 10000: # Ví dụ ngưỡng phỏng đoán là 10000
                     total_price = num_val * 1e6
                 else:
                     total_price = num_val # Giả sử đơn vị là VNĐ
            else:
                 return np.nan
        except ValueError:
            return np.nan # Trả về NaN nếu không thể chuyển đổi

    return total_price if total_price > 0 else np.nan

def extract_land_area(area_str):
    """
    Trích xuất số diện tích từ chuỗi (ví dụ: '16 m²', 'Ngang 4m x Dài 10m').
    Trả về NaN nếu không tìm thấy số.
    """
    if isinstance(area_str, (int, float)):
        return float(area_str)
    if not isinstance(area_str, str):
        return np.nan

    # Tìm số đầu tiên trong chuỗi (có thể là số nguyên hoặc số thập phân)
    match = re.search(r'(\d+\.?\d*)', area_str)
    if match:
        try:
            return float(match.group(1))
        except ValueError:
            return np.nan
    # Có thể thêm logic xử lý 'Ngang X x Dài Y' nếu cần
    # Ví dụ: match_dim = re.search(r'ngang\s*(\d+\.?\d*).*dài\s*(\d+\.?\d*)', area_str.lower())
    # if match_dim: return float(match_dim.group(1)) * float(match_dim.group(2))
    return np.nan

In [ ]:
# Xem toàn bộ dữ liệu
data = pd.read_csv(file_path)
data

In [ ]:
# --- 2. Tải và Khám phá Dữ liệu ---
# Giả sử file dữ liệu của bạn tên là 'data.csv' và có 10 cột bạn đã nêu
# **THAY ĐỔI TÊN FILE NẾU CẦN**
file_path = '/content/drive/MyDrive/real_estate_listings.csv'
# Danh sách 10 cột bạn cung cấp
expected_columns = [
    'Location', 'Price', 'Type of House', 'Land Area', 'Bedrooms',
    'Toilets', 'Total Floors', 'Main Door Direction', 'Balcony Direction',
    'Legal Documents'
]

try:
    # Đọc dữ liệu, bạn có thể cần chỉ định encoding='utf-8' nếu có lỗi
    data = pd.read_csv(file_path)
    print("Tải dữ liệu thành công!")
    data

    # Kiểm tra xem có đủ 10 cột mong đợi không
    if not all(col in data.columns for col in expected_columns):
        print("Cảnh báo: Dữ liệu không chứa đủ các cột mong đợi.")
        print(f"Các cột mong đợi: {expected_columns}")
        print(f"Các cột có trong file: {list(data.columns)}")
        # Chỉ giữ lại các cột tồn tại trong expected_columns
        data = data[[col for col in expected_columns if col in data.columns]]
        if data.empty:
             print("Lỗi: Không có cột nào phù hợp.")
             exit()

    print(f"Số dòng dữ liệu ban đầu: {data.shape[0]}")
    print("5 dòng dữ liệu đầu tiên:")
    print(data.head())
    print("\nThông tin tổng quan về dữ liệu:")
    data.info()
    print("\n Mô tả thống kê (bao gồm cả cột object):")
    print(data.describe(include='all'))


except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file tại đường dẫn '{file_path}'.")
    print("Vui lòng đặt file dữ liệu vào cùng thư mục hoặc cung cấp đường dẫn chính xác.")
    exit()
except Exception as e:
    print(f"Đã xảy ra lỗi khi đọc file CSV: {e}")
    exit()

In [ ]:
# --- 3. Tiền xử lý dữ liệu ---
print("\n Kiểm tra dữ liệu thiếu ban đầu (số lượng giá trị null):")
print(data.isnull().sum())

# 3.1 Làm sạch các cột cụ thể
price_col = 'Price'
area_col = 'Land Area'

if price_col in data.columns:
    print(f"\nBắt đầu xử lý cột '{price_col}'...")
    data[price_col] = data[price_col].apply(convert_price_v2)
    initial_rows = data.shape[0]
    data.dropna(subset=[price_col], inplace=True)
    removed_rows = initial_rows - data.shape[0]
    print(f"Đã loại bỏ {removed_rows} dòng do giá trị '{price_col}' bị thiếu hoặc không hợp lệ.")
    print(f"Số dòng dữ liệu còn lại sau khi xử lý giá: {data.shape[0]}")
else:
    print(f"Lỗi: Không tìm thấy cột '{price_col}' để làm mục tiêu.")
    exit()

if area_col in data.columns:
    print(f"\nBắt đầu xử lý cột '{area_col}'...")
    data[area_col] = data[area_col].apply(extract_land_area)
    # Giá trị diện tích NaN sẽ được xử lý bởi SimpleImputer sau
else:
    print(f"Cảnh báo: Không tìm thấy cột '{area_col}'.")
# Ép kiểu các cột số tiềm năng khác
for col in ['Bedrooms', 'Toilets', 'Total Floors']:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')
print("\n Kiểm tra dữ liệu thiếu sau khi xử lý Price/Area và ép kiểu số:")
print(data.isnull().sum())


# 3.2 Xác định đặc trưng và mục tiêu
target = price_col
# Lấy tất cả các cột còn lại làm đặc trưng tiềm năng
potential_features = [col for col in data.columns if col != target]
print(f"\nCác đặc trưng tiềm năng được sử dụng: {potential_features}")

# Tách các cột số và cột phân loại từ danh sách đã chọn
# Chuyển đổi các cột số tiềm năng sang numeric, lỗi sẽ thành NaN
for col in ['Bedrooms', 'Toilets', 'Total Floors', area_col]: # Thêm Land Area vào đây
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')

numeric_features = data[potential_features].select_dtypes(include=np.number).columns.tolist()
categorical_features = data[potential_features].select_dtypes(exclude=np.number).columns.tolist()

print(f"\nCác đặc trưng số được xác định: {numeric_features}")
print(f"Các đặc trưng phân loại được xác định: {categorical_features}")

# 3.3 Tạo Pipeline cho tiền xử lý
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

# Kết hợp các transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)],
    remainder='drop') # Bỏ qua các cột không được liệt kê (nếu có)
print("\n Cấu trúc Pipeline tiền xử lý đã được tạo.")
print(preprocessor)

# Tạo DataFrame đặc trưng X và Series mục tiêu y
X = data[potential_features]
y = data[target]

print(f"\nKích thước của X (trước tiền xử lý): {X.shape}")
print(f"Kích thước của y: {y.shape}")



In [ ]:
# --- 4. Chuẩn bị dữ liệu cho mô hình ---

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nKích thước tập huấn luyện X_train: {X_train.shape}")
print(f"Kích thước tập kiểm tra X_test: {X_test.shape}")
print(f"Kích thước mục tiêu huấn luyện y_train: {y_train.shape}")
print(f"Kích thước mục tiêu kiểm tra y_test: {y_test.shape}")

# --- 5. Huấn luyện mô hình RandomForest ---
model_pipeline_rf = Pipeline(steps=[('preprocessor', preprocessor),
                                  ('regressor', RandomForestRegressor(n_estimators=100,
                                                                     random_state=42,
                                                                     n_jobs=-1,
                                                                     max_depth=15,
                                                                     min_samples_split=10,
                                                                     min_samples_leaf=5,
                                                                     max_features=0.8))])

print("\nBắt đầu huấn luyện mô hình Random Forest Regressor...")
model_pipeline_rf.fit(X_train, y_train)
print("Huấn luyện mô hình Random Forest hoàn tất!")


# ---Huấn luyện mô hình XGBoost ---
model_pipeline_xgb = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('regressor', xgb.XGBRegressor(objective='reg:squarederror', # Mục tiêu là hồi quy lỗi bình phương
                                                                  n_estimators=100,          # Số lượng cây (iterations)
                                                                  learning_rate=0.1,         # Tốc độ học
                                                                  max_depth=7,               # Độ sâu tối đa của cây
                                                                  subsample=0.8,             # Tỷ lệ mẫu dùng cho mỗi cây
                                                                  colsample_bytree=0.8,      # Tỷ lệ đặc trưng dùng cho mỗi cây
                                                                  random_state=42,
                                                                  n_jobs=-1))])

print("\nBắt đầu huấn luyện mô hình XGBoost Regressor...")
model_pipeline_xgb.fit(X_train, y_train)
print("Huấn luyện mô hình XGBoost hoàn tất!")

In [ ]:
# --- 6. Đánh giá mô hình RandomForest ---
print("\nĐánh giá mô hình RANDOM FOREST trên tập kiểm tra:")
y_pred_rf = model_pipeline_rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest - MAE: {mae_rf:,.0f} VNĐ")
print(f"Random Forest - RMSE: {rmse_rf:,.0f} VNĐ")
print(f"Random Forest - R-squared (R²): {r2_rf:.4f}")

# ---  Đánh giá mô hình XGBoost ---
print("\nĐánh giá mô hình XGBOOST trên tập kiểm tra:")
y_pred_xgb = model_pipeline_xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"XGBoost - MAE: {mae_xgb:,.0f} VNĐ")
print(f"XGBoost - RMSE: {rmse_xgb:,.0f} VNĐ")
print(f"XGBoost - R-squared (R²): {r2_xgb:.4f}")

In [ ]:
# --- 7. Trực quan hóa và Phân tích Đặc trưng ---

#7.1 Trước khi được huấn luyện
# --- Biểu đồ cột: Phân phối Loại hình nhà ---
# Đảm bảo biến 'data' đã được tải và xử lý cột 'Price', 'Land Area'
plt.figure(figsize=(10, 6)) # Điều chỉnh kích thước biểu đồ
sns.countplot(data=data, y='Type of House', order = data['Type of House'].value_counts().index) # Sắp xếp theo tần suất giảm dần
plt.title('Phân phối số lượng theo Loại hình nhà', fontsize=14)
plt.xlabel('Số lượng', fontsize=12)
plt.ylabel('Loại hình nhà', fontsize=12)
plt.tight_layout() # Tự động điều chỉnh layout
plt.show()

# --- Biểu đồ Histogram: Phân phối Giá nhà ---
plt.figure(figsize=(15, 6))
sns.histplot(data=data, x='Price', kde=True, bins=50) # kde=True vẽ thêm đường ước lượng mật độ
plt.title('Phân phối Giá nhà (VNĐ)', fontsize=14)
plt.xlabel('Giá nhà (VNĐ)', fontsize=12)
plt.ylabel('Tần suất', fontsize=12)
plt.ticklabel_format(style='plain', axis='x') # Hiển thị số đầy đủ, không dùng ký hiệu khoa học
# Có thể cần giới hạn trục x nếu phân phối quá lệch:
# plt.xlim(0, data['Price'].quantile(0.95)) # Ví dụ: chỉ hiển thị đến phân vị 95%
plt.show()

# --- Biểu đồ Box Plot: Phân phối Giá nhà (Tổng thể) ---
plt.figure(figsize=(15, 4))
sns.boxplot(data=data, x='Price')
plt.title('Biểu đồ Box Plot cho Giá nhà (VNĐ)', fontsize=14)
plt.xlabel('Giá nhà (VNĐ)', fontsize=12)
plt.ticklabel_format(style='plain', axis='x')
# Có thể cần giới hạn trục x nếu có outliers quá lớn che mất hộp
# plt.xlim(0, data['Price'].quantile(0.99)) # Ví dụ
plt.show()

# --- Biểu đồ Box Plot: So sánh Giá nhà theo Loại hình nhà ---
plt.figure(figsize=(14, 7))
# Lấy top N loại nhà phổ biến nhất để biểu đồ không quá đông đúc (ví dụ: top 7)
top_types = data['Type of House'].value_counts().nlargest(7).index
data_top_types = data[data['Type of House'].isin(top_types)]

sns.boxplot(data=data_top_types, x='Price', y='Type of House', order=top_types)
plt.title('So sánh Phân phối Giá nhà theo Loại hình nhà (Top 7)', fontsize=14)
plt.xlabel('Giá nhà (VNĐ)', fontsize=12)
plt.ylabel('Loại hình nhà', fontsize=12)
plt.ticklabel_format(style='plain', axis='x')
# Giới hạn trục x để dễ so sánh các hộp hơn nếu cần
# plt.xlim(0, data['Price'].quantile(0.90)) # Ví dụ
plt.tight_layout()
plt.show()

# --- Biểu đồ Heatmap: Ma trận tương quan giữa các biến số ---
# Chọn các cột số để tính tương quan (bao gồm cả 'Price' đã xử lý)
numeric_cols_for_corr = data[numeric_features + [target]].copy() # Tạo bản sao để tránh thay đổi data gốc

# Tính ma trận tương quan
correlation_matrix = numeric_cols_for_corr.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.2)
# annot=True: hiển thị giá trị tương quan trên ô
# cmap='coolwarm': bảng màu (màu nóng cho tương quan dương, lạnh cho tương quan âm)
# fmt=".2f": định dạng số hiển thị (2 chữ số thập phân)
plt.title('Heatmap Ma trận Tương quan giữa các Biến số', fontsize=14)
plt.xticks(rotation=45, ha='right') # Xoay nhãn trục x để dễ đọc
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
#7.2 Sau khi được huấn luyện
# Biểu đồ Giá thực tế vs. Giá dự đoán (cho cả hai mô hình).
plt.figure(figsize=(25, 6))

plt.subplot(1, 2, 1) # Biểu đồ 1
plt.scatter(y_test, y_pred_rf, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
plt.title('Random Forest: Thực tế vs. Dự đoán')
plt.xlabel("Giá thực tế (VNĐ)")
plt.ylabel("Giá dự đoán (VNĐ)")
plt.ticklabel_format(style='plain', axis='both')

plt.subplot(1, 2, 2) # Biểu đồ 2
plt.scatter(y_test, y_pred_xgb, alpha=0.3, color='orange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
plt.title('XGBoost: Thực tế vs. Dự đoán')
plt.xlabel("Giá thực tế (VNĐ)")
plt.ylabel("Giá dự đoán (VNĐ)")
plt.ticklabel_format(style='plain', axis='both')

plt.tight_layout() # Điều chỉnh layout cho đẹp
plt.show()

# Phân tích Đặc trưng Quan trọng (cho cả hai mô hình)
def plot_feature_importance(model_pipeline, title):
    try:
        # Lấy tên đặc trưng sau khi tiền xử lý
        feature_names_out = model_pipeline.named_steps['preprocessor'].get_feature_names_out()
        # Lấy mức độ quan trọng từ bước 'regressor'
        importances = model_pipeline.named_steps['regressor'].feature_importances_
        feature_importance_df = pd.DataFrame({'feature': feature_names_out, 'importance': importances})
        feature_importance_df = feature_importance_df.sort_values('importance', ascending=False).round(4)

        print(f"\nTop 20 đặc trưng quan trọng nhất ({title}):")
        print(feature_importance_df.head(20))

        # Vẽ biểu đồ
        plt.figure(figsize=(10, 5))
        sns.barplot(x='importance', y='feature', data=feature_importance_df.head(20))
        plt.title(f'Top 20 Đặc trưng quan trọng nhất ({title})')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"\nKhông thể lấy hoặc hiển thị đặc trưng quan trọng ({title}): {e}")

# Vẽ cho Random Forest
plot_feature_importance(model_pipeline_rf, "Random Forest")
# Vẽ cho XGBoost
plot_feature_importance(model_pipeline_xgb, "XGBoost")